# Phase II — Leakage audit, stronger baseline, and dimension-controlled geometry test

This notebook starts **after** the Phase-I 512 px / $\sigma=\{1,2,4,8\}$ curvature extraction.

It addresses three reviewer-facing concerns:

1. possible train/validation duplicates or near-duplicates;
2. the weak legacy edge-density feature;
3. unequal feature dimensionality between the conventional baseline and the geometry-augmented model.

The Phase-I curvature matrices are **reused**; they are not recomputed here.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

# Always move outside the repo before clone/pull operations.
os.chdir("/content")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
print("Repository:", REPO_DIR)
print("Branch:", BRANCH)


In [ ]:
import kagglehub
from pathlib import Path

DATASET_ID = "delayedkarma/impressionist-classifier-data"
dataset_path = Path(kagglehub.dataset_download(DATASET_ID))
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def has_artist_folders(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    subdirs = [p for p in path.iterdir() if p.is_dir()]
    if len(subdirs) < 2:
        return False
    return any(
        any(f.is_file() and f.suffix.lower() in IMAGE_EXTS for f in d.rglob("*"))
        for d in subdirs[:5]
    )

def locate_split(base: Path, split: str) -> Path:
    for c in [base / split / split, base / split, base]:
        if has_artist_folders(c):
            return c
    for c in base.rglob(split):
        if c.is_dir() and has_artist_folders(c):
            return c
    raise FileNotFoundError(f"Could not locate {split} below {base}")

TRAIN_ROOT = locate_split(dataset_path, "training")
TEST_ROOT = locate_split(dataset_path, "validation")
print("TRAIN_ROOT:", TRAIN_ROOT)
print("TEST_ROOT :", TEST_ROOT)


In [ ]:
# Recover the Phase-I feature matrices without recomputing curvature.
# If they are not already present, Colab will ask you to upload
# painting_geometry_first_results.zip.

import zipfile
from google.colab import files

GEOM_TRAIN = RESULTS_DIR / "features_train_multiscale.csv"
GEOM_TEST = RESULTS_DIR / "features_test_multiscale.csv"

if not (GEOM_TRAIN.exists() and GEOM_TEST.exists()):
    print("Phase-I feature CSVs are not in this runtime.")
    print("Upload painting_geometry_first_results.zip ...")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("No ZIP uploaded.")
    zip_path = Path("/content") / zip_names[0]
    with zipfile.ZipFile(zip_path) as z:
        for target in ["features_train_multiscale.csv", "features_test_multiscale.csv"]:
            matches = [n for n in z.namelist() if n.endswith(target)]
            if not matches:
                raise FileNotFoundError(f"{target} not found in ZIP")
            with z.open(matches[0]) as src, open(RESULTS_DIR / target, "wb") as dst:
                dst.write(src.read())

import pandas as pd
geom_train = pd.read_csv(GEOM_TRAIN)
geom_test = pd.read_csv(GEOM_TEST)
print("Geometry train:", geom_train.shape)
print("Geometry test :", geom_test.shape)
print("Geometry features:", sum(c.startswith(("curv__", "orient__")) for c in geom_train.columns))


In [ ]:
# 1) Cross-split leakage audit with exact SHA1 + pHash + dHash.
# Candidates are generated at a permissive threshold <=6/<=6.
# The main clean evaluation later removes only <=4/<=4 (or exact-byte matches).

LEAK_DIR = RESULTS_DIR / "phase2_leakage"
cmd = [
    sys.executable, "-u", "scripts/audit_near_duplicates.py",
    "--train-root", str(TRAIN_ROOT),
    "--test-root", str(TEST_ROOT),
    "--output-dir", str(LEAK_DIR),
    "--phash-threshold", "6",
    "--dhash-threshold", "6",
]
subprocess.run(cmd, check=True)

display(pd.read_csv(LEAK_DIR / "leakage_audit_summary.csv"))
cand = pd.read_csv(LEAK_DIR / "cross_split_near_duplicates.csv")
display(cand.head(25))

sheet = LEAK_DIR / "near_duplicate_contact_sheet.jpg"
if sheet.exists():
    from IPython.display import display, Image
    display(Image(filename=str(sheet)))


In [ ]:
# 2) Stronger conventional appearance baseline.
# This is much cheaper than recomputing multiscale curvature.

LONG_SIDE = 512
STRONG_TRAIN = RESULTS_DIR / "strong_baseline_train.csv"
STRONG_TEST = RESULTS_DIR / "strong_baseline_test.csv"
RECOMPUTE_STRONG = True

def extract_strong(root: Path, output: Path):
    cmd = [
        sys.executable, "-u", "scripts/extract_strong_baseline_features.py",
        "--root", str(root),
        "--output", str(output),
        "--long-side", str(LONG_SIDE),
    ]
    subprocess.run(cmd, check=True)

if RECOMPUTE_STRONG or not STRONG_TRAIN.exists():
    extract_strong(TRAIN_ROOT, STRONG_TRAIN)
if RECOMPUTE_STRONG or not STRONG_TEST.exists():
    extract_strong(TEST_ROOT, STRONG_TEST)

strong_train = pd.read_csv(STRONG_TRAIN)
strong_test = pd.read_csv(STRONG_TEST)
strong_cols = [c for c in strong_train if c.startswith("strong__")]
print("Strong baseline train:", strong_train.shape)
print("Strong baseline test :", strong_test.shape)
print("Strong baseline features:", len(strong_cols))

# Verify that the replacement edge densities are not constant by construction.
edge_density_cols = [c for c in strong_cols if "edge_density" in c]
display(strong_train[edge_density_cols].agg(["mean", "std", "min", "max"]).T.head(20))


In [ ]:
# 3) Tuned, leakage-aware Phase-II experiment.
# Hyperparameters are selected ONLY inside training CV.
# Both raw and leakage-clean validation metrics are reported.
# A second comparison uses the SAME selected dimensionality (k=40).

PHASE2_DIR = RESULTS_DIR / "phase2"
cmd = [
    sys.executable, "-u", "scripts/run_phase2_experiments.py",
    "--geometry-train", str(GEOM_TRAIN),
    "--geometry-test", str(GEOM_TEST),
    "--strong-train", str(STRONG_TRAIN),
    "--strong-test", str(STRONG_TEST),
    "--output-dir", str(PHASE2_DIR),
    "--leakage-candidates", str(LEAK_DIR / "cross_split_near_duplicates.csv"),
    "--clean-phash", "4",
    "--clean-dhash", "4",
    "--matched-k", "40",
    "--cv-folds", "3",
    "--n-jobs", "-1",
]
subprocess.run(cmd, check=True)


In [ ]:
# 4) Inspect the decisive Phase-II results.

import numpy as np
import matplotlib.pyplot as plt

phase2 = pd.read_csv(PHASE2_DIR / "phase2_results.csv")
deltas = pd.read_csv(PHASE2_DIR / "phase2_deltas.csv")
meta = pd.read_csv(PHASE2_DIR / "phase2_metadata.csv")
selected = pd.read_csv(PHASE2_DIR / "phase2_selected_features.csv")

display(meta)
display(phase2.sort_values(["eval_set", "macro_f1"], ascending=[True, False]))
display(deltas)

# Compact comparison plot, raw vs leakage-clean.
plot_df = phase2[phase2["experiment"].isin([
    "B_strong_full", "G_geometry_full", "BG_combined_full",
    "B_strong_k40", "BG_combined_k40"
])].copy()

fig, ax = plt.subplots(figsize=(11, 5))
labels = plot_df["experiment"] + " | " + plot_df["eval_set"]
x = np.arange(len(plot_df))
y = plot_df["macro_f1"].to_numpy()
lo = y - plot_df["macro_f1_ci_low"].to_numpy()
hi = plot_df["macro_f1_ci_high"].to_numpy() - y
ax.bar(x, y)
ax.errorbar(x, y, yerr=np.vstack([lo, hi]), fmt="none", capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=55, ha="right")
ax.set_ylabel("Macro-F1")
ax.set_title("Phase II: strong baseline, geometry, leakage audit, and matched dimensionality")
ax.set_ylim(0, min(1.0, max(0.7, float(plot_df["macro_f1_ci_high"].max()) + 0.08)))
fig.tight_layout()
fig.savefig(PHASE2_DIR / "Figure_phase2_macroF1.png", dpi=220, bbox_inches="tight")
plt.show()

print("\nSelected k=40 features:")
display(selected.groupby("experiment").size().rename("n_selected"))
display(selected.sort_values(["experiment", "anova_f"], ascending=[True, False]).groupby("experiment").head(15))


In [ ]:
# 5) Sensitivity of conclusions to the near-duplicate exclusion threshold.
# No model is re-fit here: we only re-evaluate the saved test predictions.

from sklearn.metrics import f1_score

pred = pd.read_csv(PHASE2_DIR / "phase2_predictions.csv")
cand = pd.read_csv(LEAK_DIR / "cross_split_near_duplicates.csv")

def mask_for_threshold(t):
    if cand.empty:
        return np.ones(len(pred), dtype=bool)
    flagged = cand[
        cand["exact_bytes"].astype(bool)
        | ((cand["phash_distance"] <= t) & (cand["dhash_distance"] <= t))
    ]
    keys = set(zip(flagged["test_artist"].astype(str), flagged["test_filename"].astype(str)))
    return np.array([
        (str(a), str(f)) not in keys
        for a, f in zip(pred["artist"], pred["filename"])
    ])

rows = []
for t in [0, 2, 4, 6]:
    mask = mask_for_threshold(t)
    for model in ["B_strong_full", "G_geometry_full", "BG_combined_full", "B_strong_k40", "BG_combined_k40"]:
        rows.append({
            "threshold": t,
            "model": model,
            "n_test": int(mask.sum()),
            "macro_f1": f1_score(pred.loc[mask, "artist"], pred.loc[mask, model], average="macro", zero_division=0),
        })

sensitivity = pd.DataFrame(rows)
display(sensitivity.pivot(index="threshold", columns="model", values="macro_f1"))
sensitivity.to_csv(PHASE2_DIR / "leakage_threshold_sensitivity.csv", index=False)


In [ ]:
# 6) Package the lightweight Phase-II outputs for review.
# The large Phase-I and strong-baseline feature matrices are intentionally excluded.

import shutil

PACKAGE_DIR = Path("/content/phase2_package")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir()

for folder in [PHASE2_DIR, LEAK_DIR]:
    for p in folder.iterdir():
        if p.name in {"train_hashes.csv", "test_hashes.csv"}:
            continue
        if p.is_file():
            shutil.copy2(p, PACKAGE_DIR / p.name)

zip_path = shutil.make_archive("/content/painting_geometry_phase2_results", "zip", PACKAGE_DIR)
print("Created:", zip_path)

from google.colab import files
files.download(zip_path)
